In [ ]:
# Implementing Naive Bayes with scikit-learn (toy example)
import numpy as np
from sklearn.naive_bayes import BernoulliNB

# Define dataset
X_train = np.array([
    [0, 1, 1],
    [0, 0, 1],
    [0, 0, 0],
    [1, 1, 0]
])
Y_train = ['Y', 'N', 'Y', 'Y']

X_test = np.array([[1, 0, 1]])

# Create Naive Bayes model and predict
clf = BernoulliNB(alpha=1.0, fit_prior=True)
clf.fit(X_train, Y_train)
pred = clf.predict(X_test)
print('[scikit-learn] Prediction:', pred)

In [ ]:
# Building a Movie Recommender with Naive Bayes
import numpy as np
import pandas as pd

# Import data (ratings.csv, movies.csv)
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
print(movies.head())
print(ratings.head())

# Merge movies and ratings on movieId (optional)
df = pd.merge(ratings, movies, on='movieId')
df.head()

# Count unique users and movies
n_users = df['userId'].nunique()
n_movies = df['movieId'].nunique()
print(f"Number of users: {n_users}")
print(f"Number of movies: {n_movies}")

# Check how many times each rating value occurs
values, counts = np.unique(df['rating'], return_counts=True)
for value, count in zip(values, counts):
    print(f"Number of rating {value}: {count}")

# Create a new column 'liked' with value 1 if rating >= 4
df['liked'] = (df['rating'] >= 4.0).astype(int)

# Create a user-movie matrix (0/1 liked) using pivot_table
userId_movieId_rating = df.pivot_table(index='userId', columns='movieId', values='liked')
userId_movieId_rating.fillna(0, inplace=True)

# Fix a movie ID as target; remaining movies are features
target_movie = 2858
y = userId_movieId_rating[target_movie].astype(int)
X = userId_movieId_rating.drop(columns=[target_movie]).astype(int)

# Train/test split
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

clf = BernoulliNB(alpha=1.0, fit_prior=True)
clf.fit(X_train_split, y_train_split)

pred = clf.predict(X_test_split)
print('Confusion Matrix:\n', confusion_matrix(y_test_split, pred))
print('Accuracy:', accuracy_score(y_test_split, pred))
print('Classification Report:\n', classification_report(y_test_split, pred))

# ROC curve and AUC
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_prob = clf.predict_proba(X_test_split)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test_split, y_prob)
auc_score = roc_auc_score(y_test_split, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.2f})", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve for Movie ID {target_movie}")
plt.legend()
plt.show()

# Predict preference for a specific user
user_id = 1
x_user = X.loc[user_id].values.reshape(1, -1)
predicted_preference = clf.predict(x_user)
predicted_probability = clf.predict_proba(x_user)
print('Predicted preference:', predicted_preference)
print('Predicted probability:', predicted_probability)

# Recommend top users likely to like the target movie
def recommend_movie(target_movie, top_k=10):
    not_watched_user_ids = userId_movieId_rating.index[userId_movieId_rating[target_movie] == 0]
    not_watched_users = X.loc[not_watched_user_ids]
    not_watched_users_prob = clf.predict_proba(not_watched_users)[:, 1]

    top_k_user_indices = np.argsort(not_watched_users_prob)[-top_k:][::-1]
    top_k_user_ids = not_watched_user_ids[top_k_user_indices]

    return top_k_user_ids, not_watched_users_prob[top_k_user_indices]

movie_title = movies.loc[movies.movieId == target_movie, "title"].values[0]
print(f"\nTop users likely to like '{movie_title}':")

top_users, top_probs = recommend_movie(target_movie=target_movie, top_k=10)
for user, prob in zip(top_users, top_probs):
    print(f"User ID: {user}, Probability of liking: {prob:.4f}")

# Stratified k-fold cross validation to find best alpha
from sklearn.model_selection import StratifiedKFold

alphas = np.logspace(-2, 1, 10)  # 0.01 to 10
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

alpha_scores = []
for alpha in alphas:
    clf_cv = BernoulliNB(alpha=alpha, fit_prior=True)
    fold_scores = []

    for train_idx, val_idx in cv.split(X, y):
        X_train_fold, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val = y.iloc[train_idx], y.iloc[val_idx]

        clf_cv.fit(X_train_fold, y_train_fold)
        y_prob_val = clf_cv.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_prob_val)
        fold_scores.append(score)

    mean_score = np.mean(fold_scores)
    alpha_scores.append((alpha, mean_score))
    print(f"Alpha={alpha:.3f}, Mean AUC={mean_score:.4f}")

best_alpha, best_score = max(alpha_scores, key=lambda x: x[1])
print(f"Best alpha: {best_alpha:.3f}, AUC: {best_score:.4f}")

best_model = BernoulliNB(alpha=best_alpha)
best_model.fit(X, y)

# GridSearchCV to explore other parameters
from sklearn.model_selection import GridSearchCV

param_grid = {
    'alpha': np.logspace(-2, 1, 10),
    'fit_prior': [True, False]
}

grid_search = GridSearchCV(
    BernoulliNB(),
    param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X, y)

best_alpha = grid_search.best_params_['alpha']
best_fit_prior = grid_search.best_params_['fit_prior']
best_score = grid_search.best_score_

print(f"\nBest alpha: {best_alpha:.3f}, Best fit_prior: {best_fit_prior}, AUC: {best_score:.4f}")